# VNINDEX Yang-Zhang Volatility Regime

Plot VNINDEX and classify volatility regime using rolling percentiles of Yang-Zhang variance.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

os.environ.setdefault('NUMBA_CACHE_DIR', str(pathlib.Path('.numba_cache').resolve()))
os.environ.setdefault('MPLCONFIGDIR', str(pathlib.Path('.mplconfig').resolve()))
os.environ.setdefault('XDG_CACHE_HOME', str(pathlib.Path('.cache').resolve()))
pathlib.Path(os.environ['NUMBA_CACHE_DIR']).mkdir(parents=True, exist_ok=True)
pathlib.Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
pathlib.Path(os.environ['XDG_CACHE_HOME']).mkdir(parents=True, exist_ok=True)

ANNUALISE = 252
SYMBOL = 'VNINDEX'
WINDOW = 20
LOOKBACK = 252

plt.style.use('dark_background')
print('imports ok')

In [ ]:
LOCAL_FILE = 'stocks_data_latest.h5'
REMOTE_HOST = 'https://minio.phuchuynh.xyz'

if os.path.exists(LOCAL_FILE):
    print(f'Loading from local HDF5: {LOCAL_FILE}')
    with pd.HDFStore(LOCAL_FILE, mode='r') as store:
        df_all = store['stocks']
else:
    print('Loading from Delta Lake ...')
    from deltalake import DeltaTable
    storage_options = {
        'AWS_ACCESS_KEY_ID': 'CzOwnLkEDXQy951AOqes',
        'AWS_SECRET_ACCESS_KEY': 'fdRe91TOtqTl0icUkZLsUnWvZa90aZ5qG5rVEf7S',
        'AWS_ENDPOINT_URL': REMOTE_HOST,
        'AWS_ALLOW_HTTP': 'true',
        'AWS_EC2_METADATA_DISABLED': 'true',
        'AWS_REGION': 'us-east-1',
        'aws_conditional_put': 'etag',
    }
    dt = DeltaTable('s3://delta-table-storage/stocks', storage_options=storage_options)
    now = pd.Timestamp.now()
    df_all = dt.to_pandas(
        filters=[('date', '>=', now - pd.DateOffset(years=10)), ('symbol', '=', SYMBOL)],
        columns=['symbol', 'date', 'close', 'open', 'high', 'low', 'volume'],
    )
    df_all = df_all.set_index(['date', 'symbol']).unstack(level=1)

vn = pd.DataFrame({
    'open': df_all['open'][SYMBOL],
    'high': df_all['high'][SYMBOL],
    'low': df_all['low'][SYMBOL],
    'close': df_all['close'][SYMBOL],
}).sort_index()
vn = vn.dropna()
print(f'VNINDEX rows: {len(vn):,}')

In [ ]:
def yz_daily_components(open_: pd.Series, high: pd.Series, low: pd.Series, close: pd.Series, window: int):
    """Single-period Yang-Zhang variance components."""
    log_co = np.log(close / open_)
    log_oc = np.log(open_ / close.shift(1))
    rs = (
        np.log(high / close) * np.log(high / open_)
        + np.log(low / close) * np.log(low / open_)
    )
    k = 0.34 / (1.34 + (window + 1) / (window - 1))
    return (log_oc ** 2), (log_co ** 2), rs, k

def ht_factor(window: int) -> float:
    """Hodges-Tompkins correction factor for rolling-volatility bias."""
    if window <= 1:
        return 1.0
    return 1.0 / (1.0 - (window / ANNUALISE) + ((window**2 - 1) / (3.0 * ANNUALISE**2)))

sig_o2, sig_c2, sig_rs2, k = yz_daily_components(vn['open'], vn['high'], vn['low'], vn['close'], WINDOW)
vn['yz_var_daily'] = (sig_o2 + k * sig_c2 + (1 - k) * sig_rs2).clip(lower=0)

# Step 2: roll and annualize from daily YZ variance
vn['yz_var_roll_sum'] = vn['yz_var_daily'].rolling(WINDOW).sum()
vn['yz_vol'] = np.sqrt((ANNUALISE / WINDOW) * vn['yz_var_roll_sum']) * ht_factor(WINDOW)

# Step 3-4: past-only reference distribution and percentile-rank regime
hist = vn['yz_vol'].shift(1)
p25 = hist.rolling(LOOKBACK).quantile(0.25)
p75 = hist.rolling(LOOKBACK).quantile(0.75)
p90 = hist.rolling(LOOKBACK).quantile(0.90)

def trailing_percentile_rank(series: pd.Series, lookback: int) -> pd.Series:
    vals = series.to_numpy(dtype=float)
    out = np.full(vals.shape, np.nan)
    for i in range(lookback, len(vals)):
        cur = vals[i]
        ref = vals[i - lookback:i]
        ref = ref[np.isfinite(ref)]
        if np.isfinite(cur) and ref.size > 0:
            out[i] = (ref < cur).mean() * 100.0
    return pd.Series(out, index=series.index)

vn['pct_rank'] = trailing_percentile_rank(vn['yz_vol'], LOOKBACK)

regime = pd.Series('Normal', index=vn.index)
regime[vn['pct_rank'] < 25] = 'Low'
regime[vn['pct_rank'] > 75] = 'High'
regime[vn['pct_rank'] > 90] = 'Crisis'
vn['regime'] = regime

vn[['yz_var_daily', 'yz_var_roll_sum', 'yz_vol', 'pct_rank', 'regime']].tail()

In [ ]:
plot_df = vn.dropna(subset=['yz_vol', 'pct_rank']).copy()
p25_plot = p25.reindex(plot_df.index)
p75_plot = p75.reindex(plot_df.index)
p90_plot = p90.reindex(plot_df.index)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    row_heights=[0.62, 0.38],
    subplot_titles=(
        'VNINDEX with Yang-Zhang Volatility Regime',
        f'Yang-Zhang Volatility ({WINDOW}d, HT corrected) with Past-{LOOKBACK}d Percentile Bands',
    ),
)

fig.add_trace(
    go.Scatter(x=plot_df.index, y=plot_df['close'], name='VNINDEX', line=dict(color='white', width=1.1)),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(x=plot_df.index, y=plot_df['yz_vol'] * 100, name='YZ Vol (ann %)', line=dict(color='#f7c59f', width=1.2)),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=plot_df.index, y=p25_plot * 100, name='25th pct', line=dict(color='#66bb6a', width=1, dash='dash')),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=plot_df.index, y=p75_plot * 100, name='75th pct', line=dict(color='#ef5350', width=1, dash='dash')),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=plot_df.index, y=p90_plot * 100, name='90th pct', line=dict(color='#ff1744', width=1, dash='dot')),
    row=2, col=1,
)

fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=p75_plot * 100,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    ),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df.index,
        y=p25_plot * 100,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(100,181,246,0.12)',
        name='25-75 pct band',
        hoverinfo='skip',
    ),
    row=2, col=1,
)

color_map = {
    'Crisis': 'rgba(255,23,68,0.10)',
    'High': 'rgba(239,83,80,0.08)',
    'Normal': 'rgba(100,181,246,0.05)',
    'Low': 'rgba(102,187,106,0.08)',
}
idx = plot_df.index
reg = plot_df['regime']
start = idx[0]
cur = reg.iloc[0]
for i in range(1, len(idx)):
    if reg.iloc[i] != cur:
        fig.add_vrect(x0=start, x1=idx[i], fillcolor=color_map.get(cur, 'rgba(158,158,158,0.05)'), line_width=0, layer='below', row=1, col=1)
        start = idx[i]
        cur = reg.iloc[i]
fig.add_vrect(x0=start, x1=idx[-1], fillcolor=color_map.get(cur, 'rgba(158,158,158,0.05)'), line_width=0, layer='below', row=1, col=1)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    height=820,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=-0.18, x=0, xanchor='left'),
    margin=dict(t=70, b=110, l=60, r=40),
)
fig.update_yaxes(title_text='Index', tickformat=',.0f', row=1, col=1)
fig.update_yaxes(title_text='Annualised Vol %', ticksuffix='%', row=2, col=1)
fig.update_xaxes(matches='x', row=1, col=1)
fig.update_xaxes(matches='x', rangeslider_visible=True, row=2, col=1)

fig.show()

In [ ]:
regime_share = (plot_df['regime'].value_counts(normalize=True) * 100).round(1)
print('Regime distribution (% of available days):')
for k in ['Crisis', 'High', 'Normal', 'Low']:
    print(f'  {k:6s}: {regime_share.get(k, 0.0):5.1f}%')

latest = plot_df.iloc[-1]
print('\nLatest snapshot:')
print(f"  Date         : {plot_df.index[-1].date()}")
print(f"  Close        : {latest['close']:.2f}")
print(f"  YZ Variance (daily) : {latest['yz_var_daily']:.8f}")
print(f"  Percentile Rank     : {latest['pct_rank']:.1f}")
print(f"  YZ Vol (ann) : {latest['yz_vol']*100:.2f}%")
print(f"  Regime       : {latest['regime']}")